# Prepare Dynamic Wflow Handoff

Collect or reuse the shared Wflow warmup baseline, then prepare and QA the Wflow discharge time series consumed by SFINCS for one Event-Catalog event.

Stage Contract
- Requires: built Wflow submodels with SFINCS handoff gauges, AORC source access, and reviewed NHDPlus/STREAM-geo river geometry.
- Produces: shared Wflow warmup forcing, Wflow event forcing, dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA, and acceptance JSON.
- Next: run `c_run_example.ipynb` after the handoff status is accepted.


In [1]:
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import display

# source inputs for Wflow warmup and event forcing.
from design_events.collect_sources import collect_warmup
# event selection, SFINCS staging, and handoff readiness.
from sfincs_runs.scenarios import plan_example
# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime
# meteo forcing, discharge handoff, and acceptance gates.
from wflow_runs import (
    build_meteo,
    plan_handoff,
    plan_streamflow,
    plan_warmup,
    prepare_instates,
    prepare_handoff,
    validate_geometry,
    validate_warmup_forcing,
    validate_instates,
)
# model-physics check before dynamic handoff.
from wflow_runs.build_plan import validate_staticmaps

runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=False)
location_root = runtime.location_root
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths


location: greensboro | root: /home/grahamhults/projects/Flood-RM/locations/greensboro


## 1 - Select Event and Plan Handoff


In [2]:
EVENT_ID = None  # None -> same default event as c_run_example; or e.g. "design_0398"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)

event_id = str(example_plan.event_id)
catalog = pd.read_csv(location_root / CATALOG_PATH)
catalog["event_id"] = catalog["event_id"].astype(str)
selected = catalog[catalog["event_id"] == event_id].iloc[0]

handoff_plan = plan_handoff(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(handoff_plan)

streamflow_realization = plan_streamflow(config, location_root, event_id, catalog_path=location_root / CATALOG_PATH)
display(streamflow_realization)
if streamflow_realization["status"].eq("failed").any():
    raise RuntimeError("Selected event is not ready for rainfall-driven Wflow generation (missing rainfall member or amplification reference gage).")


event_id design_0499
window_start 2018-09-12T18:00:00
window_end 2018-09-17T18:00:00
discharge_source wflow_dynamic
state_policy shared_baseline
baseline_id baseline_90d
warmup_days 90.0
warmup_baseline_root /home/grahamhults/projects/Flood-RM/locations/...
sfincs_discharge_forcing /home/grahamhults/projects/Flood-RM/locations/...
dynamic_handoff_acceptance /home/grahamhults/projects/Flood-RM/locations/...
acceptance_status missing
Name: dynamic_wflow_handoff_plan, dtype: object

,check,status,message
0,catalog_rainfall_member,passed,member=rainfall_greensboro_72h_rank0001; scale...
1,amplification_reference_gage,passed,primary_reference_gage=02094500


## Rerun Control


In [ ]:
rerun = False


## 2 - Verify Wflow Source Geometry and Static Maps


In [3]:
river_geometry = location_root / config["collection"]["national_hydrography"]["river_geometry"]
display(validate_geometry(river_geometry, raise_on_error=False))

base_root = location_root / config["wflow"].get("base_model_root", "data/wflow/base")
qa_rows = []
for submodel in config["wflow"]["domain_set"].get("submodels", []):
    model_root = base_root / str(submodel["wflow_submodel_id"])
    if (model_root / "staticmaps.nc").exists():
        report = validate_staticmaps(model_root, river_upa_km2=config["inland_coupling"]["discharge_forcing"].get("river_upa_km2"), raise_on_error=False)
        report.insert(0, "submodel_id", str(submodel["wflow_submodel_id"]))
        qa_rows.append(report)
staticmap_qa = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame()
display(staticmap_qa)
if not staticmap_qa.empty and staticmap_qa["status"].isin(["failed", "review_required"]).any():
    raise RuntimeError("Wflow staticmap QA must pass before dynamic SFINCS handoff.")


,check,status,message
0,rivwth,passed,valid=7748; unique=4943
1,rivdph,passed,valid=7748; unique=1013
2,stream_geo_source,passed,"source_columns=['rivwth_source', 'rivdph_sourc..."


""


## 3 - Collect Shared Warmup AORC Forcing


In [4]:
baseline_plan = plan_warmup(config, location_root)
display(baseline_plan)

collect_baseline_warmup = True
if collect_baseline_warmup:
    warmup_collection = collect_warmup(config, paths, force=rerun)
    display(pd.Series(warmup_collection, name="aorc_wflow_baseline_warmup"))

warmup_forcing = validate_warmup_forcing(
    config,
    location_root,
    event_id,
    reference_time=baseline_plan["baseline_reference_time"],
    raise_on_error=False,
)
display(warmup_forcing)
if warmup_forcing["status"].eq("failed").any():
    raise RuntimeError("Collect the shared AORC warmup baseline before running dynamic Wflow handoff.")

state_bootstrap = prepare_instates(config, location_root, force=rerun)
display(state_bootstrap)
if not state_bootstrap.empty and state_bootstrap["status"].eq("failed").any():
    raise RuntimeError("Create native Wflow instate/instates.nc before event replay.")

instate_readiness = validate_instates(config, location_root, raise_on_error=False)
display(instate_readiness)
if not instate_readiness.empty and instate_readiness["status"].eq("failed").any():
    raise RuntimeError("Create or promote native Wflow instate/instates.nc from the shared warmup baseline before event replay.")


state_policy shared_baseline
baseline_id baseline_90d
baseline_reference_time 2020-11-14T00:00:00
warmup_days 90.0
warmup_start 2020-08-16T00:00:00
warmup_end 2020-11-13T23:00:00
cold_state_workflow /home/grahamhults/projects/Flood-RM/locations/...
warmup_baseline_root /home/grahamhults/projects/Flood-RM/locations/...
warmup_precip /home/grahamhults/projects/Flood-RM/locations/...
warmup_temp_pet /home/grahamhults/projects/Flood-RM/locations/...
state_input instate/instates.nc
state_output outstate/outstates.nc
base_model_root /home/grahamhults/projects/Flood-RM/locations/...
Name: wflow_baseline_warmup_state_plan, dtype: object

status reused
baseline_id baseline_90d
warmup_start 2020-08-16T00:00:00
warmup_end 2020-11-13T23:00:00
precip_nc /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_nc /home/grahamhults/projects/Flood-RM/locations/...
Name: aorc_wflow_baseline_warmup, dtype: str

,file,variable,status,message
0,/home/grahamhults/projects/Flood-RM/locations/...,precip,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...
1,/home/grahamhults/projects/Flood-RM/locations/...,temp,passed,time_min=2020-08-16T00:00:00; time_max=2020-11...


2026-06-24 19:50:33,295 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-24 19:50:33,296 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-24 19:50:33,344 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-24 19:50:33,345 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/base/greensboro_rural/wflow_sbm.toml.
2026-06-24 19:50:33,348 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/base/greensboro_rural/wflow_sbm.toml.
2026-06-24 19:50:33,716 - hydromt.hydromt_wflow.components.states - states - WARNING - CRS not found in s

,submodel_id,instate,status,message
0,greensboro_rural,/home/grahamhults/projects/Flood-RM/locations/...,prepared,native setup_cold_states timestamp=2020-08-16T...


,submodel_id,instate,status,message
0,greensboro_rural,/home/grahamhults/projects/Flood-RM/locations/...,passed,ready


## 4 - Stage Wflow Event Meteo


In [5]:
meteo_report = build_meteo(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    overwrite=rerun,
)
display(pd.Series(meteo_report, name="wflow_event_meteo_forcing"))


event_id design_0499
window_start 2018-09-12T18:00:00
window_end 2018-09-17T18:00:00
rainfall_source_nc /home/grahamhults/projects/Flood-RM/locations/...
rainfall_scale_factor 1.08878
precip_path /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_path /home/grahamhults/projects/Flood-RM/locations/...
precip_provenance /home/grahamhults/projects/Flood-RM/locations/...
temp_pet_provenance /home/grahamhults/projects/Flood-RM/locations/...
precip_written True
temp_pet_written True
Name: wflow_event_meteo_forcing, dtype: object

## 5 - Run Dynamic Wflow Handoff QA


In [6]:
run_dynamic_wflow = True

handoff_report = prepare_handoff(
    config,
    location_root,
    event_id,
    catalog_path=location_root / CATALOG_PATH,
    execute=run_dynamic_wflow,
)
display(handoff_report)


2026-06-24 19:50:39,926 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-06-24 19:50:40,146 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0499/_replay_data_catalog.yml
2026-06-24 19:50:40,194 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.2).
2026-06-24 19:50:40,194 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from /home/grahamhults/projects/Flood-RM/.venv/lib/python3.13/site-packages/hydromt_wflow/data/parameters_data.yml
2026-06-24 19:50:40,213 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-06-24 19:50:40,213 - hydromt.hydromt_wflow.components.config - config - INFO - Reading model config file from /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/base/greensboro_rural/wflow_sbm.toml.
2026-06-24 19:50:40,214 - hydrom

┌ Warning: 'header' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
┌ Warning: 'params' is not recognized as a valid field of the [input] section in the TOML, this will be ignored.
└ @ Wflow ~/.julia/packages/Wflow/mJ7Ug/src/config_init.jl:81
[ Info: Wflow version v1.0.2
[ Info: Initialize model variables for model type sbm.
┌ Info: Cyclic parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0499/greensboro_rural/staticmaps.nc.
┌ Info: Forcing parameters are provided by
└ /home/grahamhults/projects/Flood-RM/locations/greensboro/data/wflow/events/design_0499/greensboro_rural/inmaps-event.nc.
┌ Info: Set atmosphere_water__precipitation_volume_flux using netCDF variable
└ precip as forcing parameter.
┌ Info: Set atmosphere_air__temperature using netCDF variable temp as forcing
└ parameter.
┌ Info: Set land_surface_water__poten

,event_id,submodel_id,window_start,window_end,update_command,resolved_update_command,hydromt_runner_status,hydromt_runner_issue,run_command,run_output_dir,status,sfincs_discharge_forcing,sfincs_discharge_written,sfincs_discharge_source,dynamic_handoff_acceptance,acceptance_status,zero_rain_control
0,design_0499,greensboro_rural,2018-09-12T18:00:00,2018-09-17T18:00:00,hydromt update wflow_sbm /home/grahamhults/pro...,/home/grahamhults/projects/Flood-RM/.venv/bin/...,project_venv,,wflow_cli /home/grahamhults/projects/Flood-RM/...,/home/grahamhults/projects/Flood-RM/locations/...,completed,/home/grahamhults/projects/Flood-RM/locations/...,True,wflow_dynamic,/home/grahamhults/projects/Flood-RM/locations/...,accepted,/home/grahamhults/projects/Flood-RM/locations/...


## 6 - Acceptance Artifact


In [7]:
acceptance_path = location_root / handoff_plan["dynamic_handoff_acceptance"]
if acceptance_path.exists():
    display(pd.read_json(acceptance_path, typ="series"))
else:
    print("Dynamic handoff planned but not executed yet:", acceptance_path)


event_id design_0499
status accepted
discharge_source wflow_dynamic
discharge_nc /home/grahamhults/projects/Flood-RM/locations/...
checks [{'check': 'event_peak', 'status': 'passed', '...
metadata {'warmup_state_plan': {'state_policy': 'shared...
dtype: object